# Crime Severity / Arrest Predictor

**Course:** Introduction to Data Science (3rd Year, 1st Term)

**Live website:** https://crime-arrest-predictor.streamlit.app 

**Source repo:** https://github.com/Joe28321/crime-arrest-predictor

This notebook trains the two RandomForest models behind the website and evaluates them:
1. **Severity** model - predicts Low / Medium / High / Critical (test acc ~26%)
2. **Arrests** model - predicts number of arrests 0-5 (test acc ~19%)

> Run all cells in order: Runtime > Run all (or Ctrl+F9).

## 1. Setup - get the code, data and libraries

In [ ]:
import os
if not os.path.exists("crime-arrest-predictor"):
    !git clone https://github.com/Joe28321/crime-arrest-predictor.git
%cd crime-arrest-predictor
!pip install -q -r requirements.txt
print("setup done")

## 2. Load the cleaned dataset

In [ ]:
import pandas as pd
df = pd.read_csv("crime_incidents_cleaned.csv")
print(df.shape)
print(df["severity"].value_counts().to_string())
print(df["num_arrests"].value_counts().sort_index().to_string())

## 3. Shared feature engineering

Datetime is split into hour/month/year/day-of-week, `reported_online` becomes 1/0/-1, IDs and post-outcome columns (resolution, case_status) are dropped to avoid leakage.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

NUM = ["latitude", "longitude", "suspect_age", "victim_age", "property_loss_usd",
       "hour", "month", "year", "dow", "reported_online"]

def add_time_features(frame):
    frame = frame.copy()
    dt = pd.to_datetime(frame["incident_datetime"], errors="coerce")
    frame["hour"] = dt.dt.hour.fillna(12).astype(int)
    frame["month"] = dt.dt.month.fillna(6).astype(int)
    frame["year"] = dt.dt.year.fillna(2021).astype(int)
    frame["dow"] = dt.dt.dayofweek.fillna(2).astype(int)
    frame["reported_online"] = frame["reported_online"].replace(
        {"1": 1, "0": 0, "Unknown": -1, 1: 1, 0: 0}).astype(int)
    return frame

def make_pipe(cat_features):
    pre = ColumnTransformer([("num", StandardScaler(), NUM),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)])
    clf = RandomForestClassifier(n_estimators=40, max_depth=14, min_samples_leaf=4,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    return Pipeline([("pre", pre), ("clf", clf)])

DROP_BASE = ["incident_id", "address", "officer_id", "officer_first_name", "officer_last_name",
             "badge_number", "suspect_id", "suspect_first_name", "suspect_last_name",
             "victim_id", "victim_first_name", "victim_last_name", "victim_phone", "notes",
             "incident_datetime", "resolution", "case_status"]
print("helpers ready")

## 4. Severity model (Low / Medium / High / Critical)

In [ ]:
sev = add_time_features(df[df["severity"].isin(["Low", "Medium", "High", "Critical"])])
X = sev.drop(columns=DROP_BASE + ["num_arrests", "severity"])
y = sev["severity"]
CAT = [c for c in X.columns if c not in NUM]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sev_pipe = make_pipe(CAT).fit(Xtr, ytr)
pred = sev_pipe.predict(Xte)
print(f"accuracy: {accuracy_score(yte, pred):.4f}")
print(confusion_matrix(yte, pred, labels=["Low", "Medium", "High", "Critical"]))
print(classification_report(yte, pred))

## 5. Arrests model (0-5)

In [ ]:
arr = add_time_features(df)
X2 = arr.drop(columns=DROP_BASE + ["num_arrests"])
y2 = arr["num_arrests"].astype(int)
CAT2 = [c for c in X2.columns if c not in NUM]
Xtr2, Xte2, ytr2, yte2 = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
arr_pipe = make_pipe(CAT2).fit(Xtr2, ytr2)
pred2 = arr_pipe.predict(Xte2)
print(f"accuracy: {accuracy_score(yte2, pred2):.4f}")
print(confusion_matrix(yte2, pred2))
print(classification_report(yte2, pred2))

## 6. Demo - predict one incident (same inputs as the website)

In [ ]:
sample = pd.DataFrame([{
    "crime_type": "Theft", "district": "North", "city": "Riverside", "state": "NY",
    "latitude": 40.0, "longitude": -75.0, "suspect_age": 30, "suspect_gender": "Male",
    "suspect_race": "White", "victim_age": 45, "victim_gender": "Female",
    "weapon_used": "Knife", "property_loss_usd": 5000.0, "reported_online": 1,
    "hour": 14, "month": 6, "year": 2021, "dow": 3}])
print("severity:", sev_pipe.predict(sample)[0])
print("severity proba:", dict(zip(sev_pipe.classes_, sev_pipe.predict_proba(sample)[0].round(3))))
sample_arr = sample.copy()
sample_arr["severity"] = "High"
print("arrests:", arr_pipe.predict(sample_arr)[0])

## 7. Which features matter most?

In [ ]:
import matplotlib.pyplot as plt
ohe = sev_pipe.named_steps["pre"].named_transformers_["cat"]
names = NUM + list(ohe.get_feature_names_out(CAT))
imp = sev_pipe.named_steps["clf"].feature_importances_
top = sorted(zip(names, imp), key=lambda x: -x[1])[:10]
plt.figure(figsize=(8, 4))
plt.barh([n for n, _ in reversed(top)], [v for _, v in reversed(top)])
plt.title("Top 10 severity-model features")
plt.tight_layout()
plt.show()
print("Conclusion: accuracy ~26%% vs 28.6%% majority baseline - the data has almost no learnable signal (expected: synthetic dataset).")